# 📰 Homework 3: Text Analysis of New York Times Articles

## Due Date: Friday, July 10th, 11:59 PM

You must submit this assignment to Gradescope by the on-time deadline, **Friday, July 10th**, at 11:59 PM. Please read the syllabus for the Slip Day policy. No late submissions beyond what is outlined in the Slip Day policy will be accepted. **We strongly encourage you to submit your work to Gradescope several hours before the stated deadline.** This way, you will have ample time to reach out to staff for support if you encounter difficulties with submission. While course staff is happy to help guide you with submitting your assignment ahead of the deadline, we will not respond to last-minute requests for assistance.

Please read the instructions carefully when submitting your work to Gradescope.

## Collaboration Policy

Data science is a collaborative activity. While you may talk with others about the homework, we ask that you **write your solutions individually**. If you do discuss the assignments with others, please **include their names** below.


**Collaborators**: _list collaborators here_


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw03.ipynb")

## 📝 This Assignment
Welcome to Homework 3! In this assignment, we will analyze trending topics in New York Times articles from the past six years.

You will gain experience with:

- Cleaning and exploring a text-based dataset,
- Manipulating data in `pandas` using `string` accessors,
- Creating and interpreting visualizations with `seaborn`,
- Writing and applying regular expressions (regex) with `pandas`, and
- Performing sentiment analysis on text using the `DistilBERT` language model.

In [ ]:
# Run this cell to set up your notebook. 
import warnings
warnings.simplefilter(action="ignore")

import re
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure that pandas shows at least 280 characters in columns, so we can see full articles.
pd.set_option("max_colwidth", 280)
plt.style.use("fivethirtyeight")
sns.set()
sns.set_context("talk")
sns.set_palette("colorblind")

In this assignment, we will use the [DistilBERT model](https://medium.com/huggingface/distilbert-8cf3380435b5), a Natural Language Processing (NLP) model designed to capture the context and meaning of words within sentences. While you are not expected to understand the intricate details of the model, we will use it to perform sentiment analysis on textual data. The necessary tools and the corresponding model are imported below.

## ⚠️ IMPORTANT NOTE

Due to limited computing resources on DataHub, the cell below **may take a significant amount of time to run** (potentially several minutes). This may also apply to other cells later in the assignment that load and use the NLP model.

This homework also uses a large dataset which can also cause some cells to take longer to run.

**Please be patient**, wait, and **avoid restarting the kernel or rerunning these cells** more than necessary. Doing so can slow down *your* notebook and impact *other students* on the same CPU cluster.

Additionally, **DO NOT** open this assignment in multiple tabs or windows. This can cause your notebook to crash and affect DataHub's performance.

Please be patient and seek assistance during Office Hours or on Ed if you encounter any issues!

Please also **ignore** any warning messages the code may output


In [ ]:
from transformers import pipeline
model_checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

## 💯 Score Breakdown

 Question | Manual| Points
--- |---| ---
1a |No| 1
1b |No| 1
1c |No| 1
2a |No| 2
2bi |No| 1
2bii |No| 1
2biii |Yes| 1
2c |No| 2
2d |No| 2
2ei |Yes| 2
2eii |Yes| 2
2eiii | Yes | 2
3a |No| 1
3b |Yes| 1
3ci |No| 1
3cii |Yes| 1
3d |Yes| 1
4 |Yes| 2
**Total** |  | **25**

## 🏎️ Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question.

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

**Important note: The local autograder tests will not be comprehensive.** There might be hidden tests used to scoring the homework, for Summer 2026, hidden tests are provided in the [debugging guide here](https://ds100.org/debugging-guide).

Finally, unless we state otherwise, **do not use for loops or list comprehensions**. The majority of this assignment can be done using built-in commands in `pandas` and `NumPy`.

### 🐛 Debugging Guide

If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Datahub, Gradescope, and common `pandas` and RegEx errors.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 📊 Question 1: Importing the Data

The data for this assignment is sourced from the [New York Times (NYT) Archive API](https://developer.nytimes.com/docs/archive-product/1/overview), which provides information about articles published in the past.

The file `data/nyt_articles.txt` contains a random sample of filtered data of specific NYT articles published between 2019 and 2024 (inclusive). Each article discusses trending topics, which we will specify shortly.

<br>

---

### 📊 Question 1a

Let's examine the contents of the `data/nyt_articles.txt` file.

Using the [`open()` function](https://docs.python.org/3/library/functions.html#open) and the [`read()` method (documentation)](https://docs.python.org/3/tutorial/inputoutput.html#methods-of-file-objects) of a `python` file object, read **the first 200 characters** of the file `data/nyt_articles.txt` and store the result in the variable `q1a`. Then, print the result to inspect it. This may take a few seconds to run.

**CAUTION:** Viewing the contents of large files in a Jupyter Notebook can crash your browser. Be careful not to print the entire contents of the file.

In [ ]:
...
print(q1a)

In [ ]:
grader.check("q1a")

<br>

---

### 📊 Question 1b

Based on the printed output from `q1a`, what format is the data in?

**A.** CSV<br/>
**B.** JavaScript Object Notation (JSON)<br/>
**C.** HTML<br/>
**D.** Excel (XLSX)

Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, `"C"`, or `"D"`, stored in the variable `q1b`.

**CAUTION:** Viewing the contents of large files in a Jupyter Notebook can crash your browser. Be careful not to print the entire contents of the file, and do not use the file explorer to open data files directly.

In [ ]:
q1b = ...

In [ ]:
grader.check("q1b")

<br>

---

### 📊 Question 1c
`pandas` has built-in readers for many different file formats. To learn more about these, check out the documentation:

- [`pd.read_csv` (docs)](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_csv.html)
- [`pd.read_json` (docs)](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_json.html)
- [`pd.read_html` (docs)](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_html.html)
- [`pd.read_excel` (docs)](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_excel.html)

Load the file `data/nyt_articles.txt` as a `DataFrame`, and store it in the variable `news_df`.

**Hint:** If your code is taking a while to run, you should review your answers to `q1a` and `q1b`; you may have used the incorrect data loading function.

In [ ]:
...
news_df.head()

In [ ]:
grader.check("q1c")

<br/>

<hr style="border: 1px solid #fdb515;" />

##  📈 Question 2: Topic Trends Over Time

Now that we've loaded the NYT data, let's analyze trends in different topics. This will help us understand how various subjects have evolved over the years and identify any significant patterns or shifts in public interest.

We will start by extracting date and time information from the articles and then proceed to analyze the frequency and context of specific topics mentioned in the articles.


<br>

---

### 📈 Question 2a

In this question, we will process the `pub_date` column of our dataset to canonicalize time-related data.
This will help us investigate the trend of news articles across units of time like years, months, and seasons.

Write a regular expression pattern that extracts the year, month, and minute from each timestamp.
Store your regex pattern in the variable `pattern`.
Your pattern should use exactly 3 capture groups:

1. The first capture group should extract the year.
2. The second capture group should extract the month.
3. The third capture group should extract the minute.

The provided code will use your pattern with `Series.str.extract`, rename the extracted columns, and convert the results to integers.

Note: For this problem, you are not permitted to use methods from the Series.dt accessor.

*Hint 1: Use a raw string, written as `r"..."`, for your regex pattern.*

*Hint 2: You may find it helpful to copy an example timestamp into [regex101](https://regex101.com) to test your pattern.*

In [ ]:
pattern = ...
dates = (
    news_df["pub_date"]
    .str.extract(pattern)
    .rename(columns={0:"Year", 1:"Month", 2:"Minute"})
) 

dates.head()

In [ ]:
grader.check("q2a")

The cell below creates the `news_df_dates` DataFrame by adding the extracted date/time columns from `dates` to `news_df`.

Make sure to run this cell!

In [ ]:
news_df_dates = news_df.merge(right=dates, left_index=True, right_index=True)
news_df_dates.head()

<br>

---

### 📈 Question 2b

In this question, we will answer some EDA questions about `news_df_dates`.

####  Question 2b, Part i
In `news_df_dates`, suppose we create a new column `num_google_mentions` that records the number of times the word `"google"` is mentioned in each news article. What type of variable is `num_google_mentions`?

**A.** Qualitative Ordinal variable<br/>
**B.** Quantitative variable<br/>
**C.** Qualitative Nominal variable

Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, or `"C"`, stored in the variable `q2bi`.

In [ ]:
q2bi = ...

In [ ]:
grader.check("q2bi")

####  Question 2b, Part ii
Which of the following options best describes the granularity of `news_df_dates`? 

Each row in `news_df_dates` uniquely describes:

**A.** A news article. <br/>
**B.** A calendar date. <br/>
**C.** An hour of a calendar date. 


Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, or `"C"`, stored in the variable `q2bii`.

In [ ]:
q2bii = ...

In [ ]:
grader.check("q2bii")

<!-- BEGIN QUESTION -->

####  Question 2b, Part iii

Suppose we wanted to investigate trends in how often the word `"AI"` is mentioned in the news since the 1980s.

Is `news_df` a suitable dataset for this investigation? Explain your reasoning.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---

###  📈 Question 2c

Some news articles include quotes in their lead paragraph (i.e., first paragraph) to grab the reader's attention and provide additional context. For the purposes of this question, a quote is defined as a sequence of characters starting with the character `"` and ending with a comma (`,`), period (`.`), question mark (`?`), or exclamation point (`!`), followed by a closing `"`. For example:
- `"The mitochondria is the powerhouse of the cell!"`
- `"Did DATA C100 course staff host a social event with staff from DATA C8?"`
- `"R is great," A TA said, "but have you tried using Python?"`

If we follow the definition above, the following text snippet contains two quotes:
- `The TA asked, "What's the purpose of regular expressions?" The student thought for a moment and then replied, "Regex are used to identify patterns in text."`



Brandon wants to extract individual quotes from paragraphs using the definition of a quote given above.
Brandon proposes the regex pattern `r'\".*[,\.\?\!]\"'`. Unfortunately, Brandon's pattern identifies only one quote in the test string below instead of two.

Modify Brandon's regex pattern so that it correctly matches the two quotes individually. Store your new pattern in the variable `modified_pattern`.

In [ ]:
test_string = '"How are your classes?" the student asked. "Super challenging! But a lot of fun!" said their roommate.'

print("Test string:", test_string)
print("Original pattern identifies:", re.findall(r'\".*[,\.\?\!]\"', test_string))

modified_pattern = ...
print("Modified pattern identifies:", re.findall(modified_pattern, test_string))

In [ ]:
grader.check("q2c")

<br>

---

### 📈 Question 2d

Now we will count how often each article's lead paragraph mentions three topics: New Year, Zoom, and GPT-related models.

Before writing the patterns, let's introduce **[word boundaries](https://www.regular-expressions.info/wordboundaries.html)** (`\b`). A word boundary matches the edge between a word character and a non-word character. This helps avoid accidental partial matches. For example, `\bZoom\b` matches `"Zoom"` but not `"Zoomer"`.

For each topic below, write a regex pattern that matches exactly the topic definition. Store your patterns in `NewYearPattern`, `ZoomPattern`, and `GPTPattern`.

Here are the definitions of one mention:

- **New Year**: Match `"New Year"` or `"New Year's"` as a phrase. Match capitalization exactly. For example, `"Happy New Year!"` and `"New Year's Eve"` should match, but `"new year"` and `"New Years"` should not.
- **Zoom**: Match `"Zoom"` as a standalone word. For example, `"Zoom"` should match, but `"Zoomer"` and `"zoom"` should not.
- **GPT Model**: Match either:
  - letters followed by optional `-`, then `GPT`, such as `"ChatGPT"` or `"CHAT-GPT"`;
  - or `GPT-`, then one digit, then an optional letter, such as `"GPT-3"` or `"GPT-4o"`.

Use word boundaries in your patterns so that partial words do not match. For example, `"fooGPTbar"` should not match.

After you write the three patterns, the provided loop will use them to create three integer columns in `news_df_dates`: `"New Year"`, `"Zoom"`, and `"GPT Model"`.

In [ ]:
NewYearPattern = ...
ZoomPattern = ...
GPTPattern = ...

topic_pattern_map = {
    "New Year": NewYearPattern,
    "Zoom": ZoomPattern,
    "GPT Model": GPTPattern,
}

for topic, regex_pattern in topic_pattern_map.items():
    news_df_dates[topic] = (
        news_df_dates["lead_paragraph"]
        .str.findall(regex_pattern)
        .str.len()
    )

news_df_dates.head(1)

In [ ]:
grader.check("q2d")

<br>

---
### 📈 Question 2e

To carry out this question, we need to add an extra a column to `news_df_dates` called `Quarter` that contains the [quarter](https://www.investopedia.com/terms/q/quarter.asp#:~:text=The%20standard%20calendar%20quarters%20that,August%2C%20and%20September%20(Q3)) each news article was published.

Each value of `Quarter` is in the format `"<Year>Q<Quarter Number>"`.

In [ ]:
quarter_num = (news_df_dates["Month"] - 1) // 3 + 1
news_df_dates['Quarter'] = news_df_dates["Year"].astype(str) + "Q" + quarter_num.astype(str)
news_df_dates.head()

The visualization below shows the number of articles that mention each topic at least once, separately for every quarter in the data.

<center>
    <img src="./images/topic_mentions.png" width="70%" align="left" alt="Line graph with Quarter on the x-axis and Number of Articles on the y-axis. Three different colored lines are shown: New Year in blue with regular spikes, Zoom in orange with a decreasing trend, and GPT Model in green which spikes starting in 2022 and then decreasing.">
</center>


In the following two parts, you will first write an analysis plan in plain English to reproduce the dataset plotted above. Then, you will ask an LLM to translate your analysis plan in Pandas and seaborn code.

<!-- BEGIN QUESTION -->

#### Question 2e, Part i

Consider the single DataFrame that was used to plot the visualization above using Seaborn.

Write an analysis plan that transforms `news_df_dates` into the DataFrame needed to make the plot.

Use the canonical English terms we learned in lecture. For example, you might use 
"for each", "sort by" or "order by", "filter to", "select first n rows", "select nth row", "join", "count rows", "sum", or "average".

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

#### Question 2e, Part ii

Provide your analysis plan to an LLM, making to sure to ask the LLM to translate your plan into Pandas and seaborn code. Paste the Python code generated by the LLM below.

 Your code should compute the quarterly topic counts and produce a plot similar to the one above. The plot does not have to be formatted exactly the same, but it should show the same pattern.

 *The real-life connection: You often visualize the plot you want in your head, and then describe it to an LLM just as you will do here!*

In [ ]:
...

# DO NOT MODIFY THE CODE BELOW
# If your solution above is correct, running this cell should produce the plot above.
plt.xticks(rotation=60)
plt.yticks()
plt.ylabel("Number of Articles")
plt.xlabel("Quarter")
plt.title("Number of Articles Released (2019-2024)")
plt.gcf().set_facecolor('white')
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

####  Question 2e, Part iii

For each of the three topics, identify one interesting pattern in the visualization and provide a tentative explanation of why you think the pattern exists.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 1px solid #fdb515;" />

## ⁉️ Question 3: Sentiment Analysis

**Sentiment analysis** involves using an NLP model to classify the emotional tone of text. For example, "You're great!" has a positive sentiment, while "I feel horrible" has a negative sentiment.

In this section, we will explore temporal changes in the **sentiment** of NYT articles that mention each topic.

> The sentiment values in this section were computed using a fine-tuned version of the **DistilBERT** model ([original paper](https://arxiv.org/abs/1910.01108)).
>
> DistilBERT is a neural network-based language model similar to ChatGPT. These models are not covered in Data 100, and we don't expect you to know how they work. If you're interested in learning more, consider taking `CS182: Neural Networks` or `Data 102: Data, Inference, and Decisions`.
>
> The [HuggingFace library](https://huggingface.co/) was used to build the sentiment analysis pipeline and load the model. [Here is the **model card** of the `DistilBERT` model we used.](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) The model card contains general information about the model, such as training arguments and training data. Again, you don't need to know these details for Data 100!

Run the following cells to set up the sentiment analysis pipeline and see examples of how we can get the sentiment for different strings.

**Note**: you may receive some warning message. Feel free to ignore this.


In [ ]:
# Load the model
sentiment_analysis = pipeline("sentiment-analysis", model=model_checkpoint)

In [ ]:
# Get the sentiment of a given string
sentiment_1 = sentiment_analysis("I have two dogs.")
print("Example 1: " + str(sentiment_1))

sentiment_2 = sentiment_analysis("I do not have dogs.")
print("Example 2: " + str(sentiment_2))

sentiment_3 = sentiment_analysis("Fortunately, I do not have dogs to worry about.")
print("Example 3: " + str(sentiment_3))

As you can see, the model returns a label (`POSITIVE` or `NEGATIVE`) and a confidence score from 0 to 1. Later, we convert this into a signed sentiment score by keeping positive-label scores positive and making negative-label scores negative.

**Note:** The output is a list, and each list element is a dictionary with two keys (label and score). Note that we could have gotten the sentiments of the three sentences by putting them in a list (batch) and then running the pipeline once (see the code below).


<br>

---
### ⁉️ Question 3a

Try it out yourself! The sentences we provided in the previous example have pretty high confidence scores. Let's see how the model behaves with more ambiguous sentences.

Write a sentence `low_confidence_sentence` for which the model's confidence score is less than 0.8.

In [ ]:
low_confidence_sentence = ...
results = sentiment_analysis(low_confidence_sentence)
print(results)

In [ ]:
grader.check("q3a")

<br>

---

### Getting the Sentiment Data

Having all 1000+ students in Data 100 run the `DistilBERT` model for all articles is too computationally intensive for DataHub. So, we have done this part for you.

1. We have loaded the sentiment scores from `nyt_sentiments.csv`.
2. We have added a new column called `article_sentiment` that provides the existing score if the label is "POSITIVE", and negates the existing score if the label is "NEGATIVE". For example, a score of `0.6` with a label `POSITIVE` would have the value `0.6` in `article_sentiment`, while a score of `0.6` with a label `NEGATIVE` would have the value `-0.6` in `article_sentiment`.
3. We have merged `news_df_dates` with `sentiment` using the `web_url` column. The merged `DataFrame` is called `news_df_sentiment`.

In [ ]:
# DO NOT EDIT THIS CELL
sentiment = pd.read_csv("data/nyt_sentiments.csv")

sentiment["article_sentiment"] = (
    (2*(sentiment["label"] == "POSITIVE") - 1) * sentiment["score"]
)
news_df_sentiment = news_df_dates.merge(sentiment, on="web_url")
news_df_sentiment = news_df_sentiment.drop(columns=["label", "score"])

Take a look at this new `DataFrame` called `news_df_sentiment`.

In [ ]:
news_df_sentiment.head(3)

Now, to continue our analysis from Question 2, let's focus on the articles that mention `Zoom`, `New Year`, or `GPT`. We have created a filtered `DataFrame` called `news_df_filtered` that contains all the information from `news_df_sentiment` with only the articles that mention `Zoom`, `New Year`, or `GPT`. We use the topic-count columns from Q2d: an article is included if at least one of `Zoom`, `New Year`, or `GPT Model` is greater than 0.

In [ ]:
news_df_filtered = news_df_sentiment[((news_df_sentiment['Zoom'] > 0) | news_df_sentiment['New Year'] > 0) | (news_df_sentiment['GPT Model'] > 0)]

Take a look at this new `DataFrame` called `news_df_filtered`.

In [ ]:
news_df_filtered.head(3)

<!-- BEGIN QUESTION -->

<br>

---
### ⁉️ Question 3b

Let's now visualize the distribution of article sentiment.

Using `seaborn`, we created a histogram to visualize the distribution of `article_sentiment`. Run the cell below to display the plot.

In [ ]:
sns.histplot(data=news_df_filtered, x='article_sentiment')
plt.xlabel('Sentiment of Leading Paragraphs')
plt.title('Histogram of Leading Paragraph Sentiment')
plt.plot();

Are you at all surprised by the distribution of sentiment in the graph above? Describe what you notice about the graph.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---
### ⁉️ Question 3c

Let's audit our data to better understand how well the sentiment analysis model works with our specific dataset. It's good practice to compare our assumptions to model outputs.

####  Question 3c, Part i

Using `news_df_filtered`, assign `top_positive` and `top_negative` to `DataFrame`s containing the five articles with the highest `article_sentiment` values and the five lowest `article_sentiment` values, respectively. The `DataFrame`s should have the columns `lead_paragraph` and `article_sentiment`.

In [ ]:
top_positive = ...
top_negative = ...

display(top_positive, top_negative)

In [ ]:
grader.check("q3ci")

<!-- BEGIN QUESTION -->

####  Question 3c, Part ii

Do you agree with the current sentiment-based ordering of news articles, or would you rearrange the ordering? Do you feel that the DistilBERT model is a good model for our task of analyzing sentiment in news articles?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---

### Continued Visualizing
Let's continue to visualize the `news_df_filtered` data. The cell below adds a new datetime column `date` to `news_df_filtered`. The datetime format will make visualization easier.

In [ ]:
# Combine the columns into a single date string in 'YYYY-MM-DD' format
news_df_filtered['date_str'] = (
    news_df_filtered['Year'].astype(str)
    + '-' + news_df_filtered['Month'].astype(str)
    + '-' + news_df_filtered['pub_date'].str[8:10]
)

# Convert the combined string to a datetime object using pd.to_datetime()
news_df_filtered['date'] = pd.to_datetime(news_df_filtered['date_str'], format='%Y-%m-%d', errors='coerce')

Below, we visualize the change in sentiment in the topic `Zoom` over time, using `sns.lineplot` to plot `date` on the x-axis and `article_sentiment` on the y-axis.

**Note**: If the following plot is empty, please rerun from all cells before Part 3b where `news_df_sentiment` was initialized.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=news_df_filtered[news_df_filtered["Zoom"] > 0], x='date', y='article_sentiment')
plt.xticks(rotation=70);

**This plot is not very pretty!** This isn't because of any errors on your part. Instead, we need to use a different visualization method to understand our data.

<!-- BEGIN QUESTION -->

### ⁉️ Question 3d

Let's visualize our data more effectively. We will still use `sns.lineplot`, but instead of plotting every observation, we will first aggregate our data, and then plot the aggregated values. We will compare sentiment scores across three topics: `New Year`, `Zoom`, and `GPT`.

You do not need to write any code for this question, just run the code cell below and answer the question below the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
dfs_per_topic = []
topics = ["New Year", "Zoom", "GPT Model"]
for topic in topics:
    df_of_current_topic = news_df_filtered[news_df_filtered[topic] > 0].groupby("Quarter")["article_sentiment"].mean().reset_index()
    df_of_current_topic["Topic"] = topic
    dfs_per_topic.append(df_of_current_topic)

all_topic_qtr_avg_sentiments = pd.concat(dfs_per_topic)
sns.lineplot(data=all_topic_qtr_avg_sentiments, x="Quarter", y="article_sentiment", hue="Topic")

plt.title('Avg. Sentiment per Topic Across Quarters')
plt.xlabel('Time')
plt.ylabel('Lead Paragraph Sentiment')

plt.axhline(0, color='black')
plt.xticks(rotation=65);

In 1-2 sentences, identify one interesting pattern in the visualization and provide a tentative explanation why you think this pattern exists.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

---

## 🤖 Question 4: Open-Ended Question

Welcome to the **open-ended question**!

> If you have any feedback on this open-ended question, or any other homework question in Data 100, we encourage you to share your thoughts using the [content feedback form](https://forms.gle/XKa8rDYdRcQkL7zAA). You can also post to Ed.

Grading on open-ended questions is simple: **Clear evidence of thoughtfulness and effort will always receive full credit**. If your response is especially well-developed or creative, we may ask for permission to share it with the rest of the class so others can be inspired by your work! Underdeveloped ideas will receive half credit. Trivial or missing responses will receive no credit. We expect the vast majority of students to receive full credit.

**SETUP**: You are a data scientist working for the New York Times. Your manager is concerned that the New York Times exhibits **bias** in its articles and asks you to look into this.

**TASK**: Your task is to provide evidence for or against the hypothesis that the New York Times exhibits **bias** in its articles.

> Food for thought: Assume your manager has a ChatGPT Pro subscription and could easily ask the question above without reaching out to you for help. What's your added value over their ChatGPT Pro subscription? 

> Also, you might be wondering: What is the definition of "**bias**"? This is up to you to define and defend. There is no single correct answer. You might also be wondering: Who is the subject of the "**bias**"? Once again this is up to you to define.  Feel free to pick a domain that is interesting to you. Remember that a major part of data science is defining what success actually means.

For this task, you may use: 
- The `news_df_sentiment` `DataFrame`.
- Any `pandas`, `regex`, `matplotlib`, `seaborn` covered in class.
- (Optional) External resources (e.g., AI/LLMs, websites, datasets, or other Python libraries/packages).


For this question, you are allowed to [vibe code](https://en.wikipedia.org/wiki/Vibe_coding). In other words, the code you use to generate responses can be generated by a large-language model (LLM), like Gemini or ChatGPT. However, the most important component of this question is not the code—it's the presentation and persuasiveness of your results. **If you copy-and-paste default output from an LLM on this question, there is a good chance that your submission will look identical or near-identical to many other students**. While we expect many answers to this question to have similarities, obvious default output will receive no credit. Spend time thinking about the presentation of your results. 

> **Disclaimer**: As Data Science students, you should be aware of important limitations and broader considerations when it comes to the use of LLMs. 
> - LLMs do not guarantee factual accuracy and they are known to hallucinate (generate fabricated or misleading information). 
> - LLMs are trained on large datasets that can reflect and reproduce biases in race, gender, culture, and ideology. 
> - The use of LLMs may involve the sharing of sensitive and personal information with third-party services.



**Your answer should consist of the following**:
1. A **single visualization**. The contents of this visualization are ultimately up to you. As a starting point, you might think about how you can reflect your chosen definition of "**bias**". 
2. A write up of **4–10 sentences** stating your definition of "**bias**", your answer to the hypothesis, an explanation of why you have come to this conclusion, and **one possible counterargument** to your answer. Your explanation must reference the visualization. Make it obvious and clear why your visualization supports your conclusion.

**Note**: A table is not a visualization!

> Keep in mind that your manager is interested in your answer to the hypothesis and the evidence you collected to support that recommendation. Don't spend more than a sentence or two explaining how you generated your visualization (e.g., no need to write "I used the X library to construct the Y-axis in my visualization."). 


> Focus on answering your manager's question, rather than explaining the detailed steps needed to answer the question (e.g., "The Y-axis in my visualization shows Z, which is why I concluded X.").


> **IMPORTANT**: If you have any questions, please read through the [**FAQS**](https://docs.google.com/document/d/1DLzA-kaj1n87uqcImwquBOLUcXBSF6JIoY2PMywwdhU/edit?usp=sharing) first. If you can't find the answer to your question there, feel free to ask your question on Ed.

### SUBMISSION INSTRUCTIONS

**Please read these carefully**
1. In the 2 scratch cells below, feel free to write any code you need to create your **single visualization**. Please use Scratch Cell 1 for imports, and Scratch Cell 2 for plotting code to avoid issues with plot generation.
2. Once you are happy with your visualization, remove any calls to `plt.show()` as this prevents the visualization from being saved as an image.
> `plt.savefig("final_viz.png", dpi=300, bbox_inches="tight")` at the bottom of scratch cell 2 saves your visualization to `final_viz.png`.
3. Run the scratch cells one last time without `plt.show()`.
4. **Comment out all code** in both scratch cells. **Including** `plt.savefig("final_viz.png", dpi=300, bbox_inches="tight")`.
5. Run the next cell to display your visualization. Make sure this is the correct visualization you want as part of your final deliverable.
6. Set `commented_out` to `True`.
7. In the markdown cell, complete your write-up in 4–10 sentences.

**Note 1**: `news_df_sentiment` is a very large `DataFrame`. If you run into memory issues, try to filter some rows or columns before running any complex functions. Please do not make multiple copies of `news_df_sentiment` as this may also cause memory issues. If you run into memory issues please feel free to post on Ed or come to OH.

**Note 2**: When you submit, please make sure your visualization and write-up appear under Question 4 of Homework 3 Written in Gradescope.

<!-- BEGIN QUESTION -->



In [ ]:
# SCRATCH CELL 1
# (Optional) Feel free to import any extra libraries or packages here
# However, you do not need to import any extra libraries to answer the question and receive full credit
# Remember to comment out your code for step 3
...

In [ ]:
# SCRATCH CELL 2
# Feel free to do your rough work here
# Remember to comment out your code for step 5

...

# Remember to comment out this code for step 5
# Until then, do not edit this code
plt.savefig("final_viz.png", dpi=300, bbox_inches="tight")

In [ ]:
# DO NOT EDIT THIS CELL
# Run this cell and make sure the image that appears is the visualization you want

from IPython.display import Image

Image("final_viz.png")

In [ ]:
# Set commented_out = True once you have commented out all your code from the scratch cells

commented_out = ...

In [ ]:
grader.check("q4")

<!-- END QUESTION -->

<br>

---

## Takeaways

In this homework, we used a language model to evaluate the sentiment of news articles and quantify text data (qualitative data) so that we could perform data analysis on a large set of journalism data. Though we used the [HuggingFace DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) model, there are thousands of language models available for use, and with rapid innovations in the NLP research space, there are new models frequently being created. In fact, we were using a different model for this homework one year ago, which reflects how quickly the NLP field progresses. 

Different models evaluate sentiment differently. You may have noticed that the DistilBERT model struggles with evaluating neutral sentences and often gives sentences a high confidence score. When evaluating which models to use in your projects, it's useful to test them on small inputs of data to see how they perform, like we did by testing out various sentences! Different models may perform differently (often due to how the model was trained and created), so it's important to understand these differences when deciding what model to use for your data.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## You have finished Homework 3!

## Course Content Feedback

If you have any feedback about this assignment or about any of our other weekly assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://docs.google.com/forms/d/e/1FAIpQLSeNCHbTUipouL3yWFeRdl9Blwu-U4zSciTi4Kcb1p5ykCtb-A/viewform). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

## Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Once you submit this file to the HW 3 Coding assignment on Gradescope, Gradescope will automatically submit a PDF file with your written answers to the HW 3 Written assignment. If you run into any issues when running this cell, feel free to check this [section](https://ds100.org/debugging-guide/autograder-gradescope/#why-does-the-last-grader-export-cell-fail-if-all-previous-tests-passed) in the Data 100 Debugging Guide.

**Important**: Please check that your written responses were generated and submitted correctly to the HW 3 Written Assignment.

**You are responsible for ensuring your submission follows our requirements and that the PDF for HW 3 written answers was generated/submitted correctly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.

**Note**: This homework's zip file may take longer to run, so please be patient

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)